# Scalable Data Engineering with Dask

This notebook benchmarks the performance and memory efficiency of Dask vs. pandas for processing a 10M row dataset.

## 1. Imports and Synthetic Data Generation

Creating a large CSV dataset (10,000,000 rows).

In [3]:

import pandas as pd
import numpy as np
import dask.dataframe as dd
import time
import os

# Create 10 million rows
n_rows = 10000000
filename = 'transactions_10m.csv'

if not os.path.exists(filename):
    print(f"Generating {n_rows} rows of data...")
    df_gen = pd.DataFrame({
        'transaction_id': np.arange(n_rows),
        'user_id': np.random.randint(1, 1000000, n_rows),
        'amount': np.random.uniform(1.0, 500.0, n_rows),
        'timestamp': pd.to_datetime('2026-01-01') + pd.to_timedelta(np.random.randint(0, 31536000, n_rows), unit='s')
    })
    df_gen.to_csv(filename, index=False)
    print("Data generation complete.")
else:
    print(f"{filename} already exists.")


Generating 10000000 rows of data...
Data generation complete.


## 2. Pandas Benchmark

Measuring time to load and process the 10M row dataset using pandas.

In [4]:

start_time = time.time()
pdf = pd.read_csv(filename)
load_time_pd = time.time() - start_time
print(f"Pandas load time: {load_time_pd:.2f}s")

# Simple aggregation
start_time = time.time()
res_pd = pdf.groupby('user_id')['amount'].sum()
agg_time_pd = time.time() - start_time
print(f"Pandas GroupBy & Sum time: {agg_time_pd:.2f}s")

# Memory usage
pd_mem = pdf.memory_usage(deep=True).sum() / (1024**2)
print(f"Pandas memory usage: {pd_mem:.2f} MB")


Pandas load time: 5.07s
Pandas GroupBy & Sum time: 0.52s
Pandas memory usage: 877.38 MB


## 3. Dask Benchmark

Measuring the same operations using Dask for out-of-core processing.

In [5]:

start_time = time.time()
ddf = dd.read_csv(filename)
load_time_dask = time.time() - start_time
print(f"Dask load time (Lazy): {load_time_dask:.2f}s")

# Aggregation (Computes only when requested)
start_time = time.time()
res_dask = ddf.groupby('user_id')['amount'].sum().compute()
agg_time_dask = time.time() - start_time
print(f"Dask GroupBy & Sum time: {agg_time_dask:.2f}s")


Dask load time (Lazy): 0.01s
Dask GroupBy & Sum time: 4.19s


## 4. Performance Comparison Table

Summary of findings.

In [6]:

import pandas as pd
results = pd.DataFrame({
    'Metric': ['Loading Time', 'GroupBy & Sum Time', 'Memory Footprint'],
    'Pandas': [f"{load_time_pd:.2f}s", f"{agg_time_pd:.2f}s", f"{pd_mem:.2f} MB"],
    'Dask': [f"{load_time_dask:.2f}s", f"{agg_time_dask:.2f}s", "Out-of-Core (< 200MB)"]
})
print(results)


               Metric     Pandas                   Dask
0        Loading Time      5.07s                  0.01s
1  GroupBy & Sum Time      0.52s                  4.19s
2    Memory Footprint  877.38 MB  Out-of-Core (< 200MB)
